# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # Access as object, not as dict

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore dataset structure and record sets
print("\nAll available record sets and their @id fields:")
record_set_ids = []
for recordset in metadata.recordSets:
    print(f"- Name: {recordset.name}")
    print(f"  @id: {recordset['@id']}")
    record_set_ids.append(recordset['@id'])

    # Print available fields in each record set
    print("  Fields (by @id and name):")
    for field in recordset.fields:
        print(f"    - {field['@id']}: {field.name} (dataType: {field.dataType if hasattr(field,'dataType') else 'N/A'})")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this exploration, select the main record set containing the mass spectrometry table
main_record_set_id = None
for recordset in metadata.recordSets:
    # Select the record set with most fields (typically the tabular data)
    if main_record_set_id is None or len(recordset.fields) > len([r for r in metadata.recordSets if r['@id'] == main_record_set_id][0].fields):
        main_record_set_id = recordset['@id']

# Extract data from all record sets into DataFrames
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records from record set {rsid}")

print(f"\nColumns available in main record set ({main_record_set_id}):")
display_columns = dataframes[main_record_set_id].columns.tolist()
print(display_columns)
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Choose a numeric field to analyze (based on column names, e.g., 'Molecular Weight' or abundance field)
candidate_numeric_fields = [c for c in display_columns if ('weight' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower() or 'coverage' in c.lower() or 'peptide' in c.lower())]
if not candidate_numeric_fields:
    print("No apparent numeric field found. Check column names.")
numeric_field = candidate_numeric_fields[0]

# Filter: e.g., keep proteins with value above median in selected numeric field
df = dataframes[main_record_set_id]
threshold = df[numeric_field].median()

print(f"Filtering proteins with {numeric_field} > {threshold:.2f}")
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered {len(filtered_df)} records.")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present
group_fields = [c for c in display_columns if 'category' in c.lower() or 'class' in c.lower() or 'type' in c.lower() or 'description' in c.lower()]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Boxplot for numeric field (if there is a category to group by)
if group_fields:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides labeled records of mass spectrometry-identified proteins in human extracellular vesicles from mast cells.
- Data includes numeric and descriptive fields, enabling filtering, normalization, and group-wise analysis.
- Numeric fields such as protein abundance or molecular weight (depending on schema) reveal distributional properties and, if present, categorical organization.
- The data can be used for downstream analysis or ML tasks, such as classification or regression, on protein properties or experiment outcomes.